# Predicting VMAT gamma passing rates: cleaned dataset

This notebook starts from the **merged and cleaned dataset** used in the original study, so you can go
straight to analysis and modelling. Use it if you could not finish merging the raw tables, or to check
your own pipeline against it.

The column dictionary for the raw tables is in the repository `README.md`. The extra columns of the
cleaned dataset are described below.

> The data are pseudonymised clinical data. Use them only for this course; do not redistribute them.

## 0. Setup

In [ ]:
%pip install -q pyzipper gdown

In [ ]:
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyzipper

pd.set_option("display.max_columns", 60)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Download and unlock the cleaned dataset

The passkey of this archive is different from the passkey of the raw data. You get it during the lecture.

In [ ]:
import gdown


def download_and_unlock(drive_url, zip_name):
    """Download an encrypted archive from Google Drive and extract it into DATA_DIR."""
    zip_path = Path(zip_name)
    if not zip_path.exists():
        gdown.download(drive_url, str(zip_path), fuzzy=True, quiet=False)
    passkey = getpass.getpass(f"Passkey for {zip_name}: ")
    with pyzipper.AESZipFile(zip_path) as z:
        z.setpassword(passkey.encode("utf-8"))
        z.extractall(DATA_DIR)
    print("Files in data/:", sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
CLEAN_URL = "https://drive.google.com/file/d/1WEb47KneHhhN_80sPcZWjL_jqdr_OBKp/view?usp=sharing"

download_and_unlock(CLEAN_URL, "srs_gpr_clean.zip")

In [ ]:
clean = pd.read_excel(DATA_DIR / "srs_gpr_clean.xlsx")
print(clean.shape)
clean.head()

## 2. What is in the cleaned dataset

One row per arc. These are **multi-lesion** HyperArc SRS plans with a portal dosimetry analysis at
**3% / 1 mm, 20% threshold**. The dataset contains the columns of `BeamInfoHyperArc`, `PDInfo` and
`BeamComplexity` (see `README.md`) plus:

| Column | Description |
|---|---|
| `Course_y`, `Plan_y`, `MachineName_y`, `Energy_y`, `DosePerFraction_y` | Duplicates created by the merge (suffix `_y`). |
| `GPR` | **Target**, here in **percent** (0–100). |
| `BeamArea` | Maximum beam area defined by the jaws = `FX × FY / 100` (cm²). |
| `Date` | Date of the matched MPC. Empty if there was no MPC that day. |
| `BeamCenterShift`, `BeamOutputChange`, `BeamUniformityChange`, `CollimationRotationOffset`, `MLCMaxOffsetA`, `MLCMaxOffsetB`, `MLCMeanOffsetA`, `MLCMeanOffsetB`, `CouchRtnLarge`, `RotationInducedCouchShiftFullRange` | **Mean** of all MPC runs of that day on that linac. |
| `<MPC column>_count`, `_min`, `_max` | Number of runs, minimum and maximum of that MPC test on that day. |
| `total_measurements` | Number of MPC runs on that day. |
| `IsMultipleLesion` | True for all rows (multi-lesion plans only). |
| `NumberOfLesions` | Number of lesions treated by the plan (counted manually). |

Suggested starting features:

In [ ]:
APCM_FEATURES = [
    "MU", "MU/DoseFx", "beam-on time", "Arclength", "FX", "FY", "BeamArea",
    "MeanMLCSpeed", "MLCSpeedModulation", "MeanDoseRate", "DoseRateModulation",
    "MeanGantrySpeed", "GantrySpeedModulation", "Q1MLCGap", "MedianMLCGap",
    "SAS10", "MeanTGI", "MCS", "MITotal", "BI", "BM", "EdgeMetric", "LT/AL",
]
MPC_FEATURES = [
    "BeamCenterShift", "BeamOutputChange", "BeamUniformityChange", "CollimationRotationOffset",
    "MLCMaxOffsetA", "MLCMaxOffsetB", "MLCMeanOffsetA", "MLCMeanOffsetB",
    "CouchRtnLarge", "RotationInducedCouchShiftFullRange",
]
df_model = clean

In [ ]:
# TODO: how many arcs have no MPC on the delivery day? How will you handle them?

## 3. Exploratory analysis

- What does the GPR distribution look like? Is it skewed? How many arcs fall below a clinical action level?
- Which aPCMs correlate with the GPR? Which aPCMs are **strongly correlated with each other**?
- How do the MPC results change over time and between the two linacs?
- Does the GPR depend on the machine, the energy, or the number of lesions?

In [ ]:
# TODO: plots

## 4. Modelling

**Important:** each patient has several arcs, and arcs of the same patient are similar. Split the data
**by patient** (`GroupKFold`) so the test set contains only patients the model has never seen.

Start from the baseline below, then try other models and feature sets:

1. aPCMs only
2. aPCMs + MPC
3. Feature selection or dimensionality reduction (many aPCMs are redundant)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold, cross_validate


def evaluate(model, X, y, groups, n_splits=5):
    # Patient-grouped cross-validation: returns mean ± std of MAE, RMSE and R²
    cv = GroupKFold(n_splits=n_splits)
    scores = cross_validate(
        model, X, y, groups=groups, cv=cv,
        scoring={"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"},
    )
    out = {}
    for k in ["MAE", "RMSE", "R2"]:
        v = scores[f"test_{k}"] * (1 if k == "R2" else -1)
        out[k] = f"{v.mean():.3f} ± {v.std():.3f}"
    return out

In [ ]:
d = df_model.dropna(subset=APCM_FEATURES)
X, y, groups = d[APCM_FEATURES], d["GPR"], d["PatientID"]
evaluate(DummyRegressor(strategy="mean"), X, y, groups)

In [ ]:
# TODO: try other models (linear regression with scaling, random forest, gradient boosting, ...)

## 5. Interpretation

- Which features matter most (permutation importance, SHAP, coefficients)?
- Does adding MPC improve the prediction over aPCMs alone?
- Is the error small enough to be clinically useful? Could the model flag arcs likely to fail?

In [ ]:
# TODO